In [111]:
import os

from dotenv import load_dotenv
load_dotenv("./.env")

from google import genai

import chromadb

from IPython.display import Markdown, display

import urllib.request
from bs4 import BeautifulSoup

## Embedding the query

In [112]:
query = "Bardzo boli mnie głowa i potrzebuje szybko zabić ból, mam uczulenie na salicyl, więc tak żebym się nie przekręcił"

In [113]:
from google.genai.models import types
vector = None

try:
    client = genai.Client(
        api_key=os.environ["GOOGLE_API_KEY"]
    )

    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[query],
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY"
        )
    )

    vector = response.embeddings[0].values

except Exception as e:
    print(e)

## Database quering

In [114]:
client_chroma = chromadb.PersistentClient("./chroma_db")
collection =client_chroma.get_collection("documents")

In [115]:
results = collection.query(
    query_embeddings=[vector],
    n_results=3
)
documents_rag = [doc for doc in results['documents'] ]
print(results["metadatas"])

[[{'med_name': 'Alka-Seltzer®'}, {'med_name': 'Almozen'}, {'med_name': 'Allegra'}]]


In [116]:
def rag_query(query, context):
    prompt = f"""
        Jesteś asystentem medycznym działającym w systemie RAG.

        ZASADY PODSTAWOWE:
        - Odpowiadasz WYŁĄCZNIE na podstawie dostarczonego kontekstu.
        - Jeśli w kontekście nie ma informacji potrzebnych do odpowiedzi, napisz: "brak danych".
        - Nie używasz wiedzy spoza kontekstu.
        - Nie informujesz użytkownika o istnieniu kontekstu ani systemu RAG.

        ZASADY DOTYCZĄCE LEKÓW:
        - Jeśli w kontekście znajdują się leki, możesz je opisać i porównać.
        - Nie pomijasz leków tylko dlatego, że są mniej odpowiednie — przedstawiasz je obiektywnie.
        - Możesz wskazać, które opcje wydają się bardziej adekwatne w kontekście objawów, jeśli wynika to bezpośrednio z danych w kontekście.

        DAWKOWANIE I BEZPIECZEŃSTWO:
        - Informacje o dawkowaniu, przeciwwskazaniach i działaniu podawaj WYŁĄCZNIE jeśli są obecne w kontekście (np. w ulotkach).
        - Nie tworzysz żadnych nowych dawek ani zaleceń.

        STRUKTURA ODPOWIEDZI:
        - Najpierw krótka analiza objawów (jeśli możliwa z kontekstu)
        - Następnie lista możliwych leków z kontekstu
        - Przy każdym leku: działanie, wskazania, przeciwwskazania (jeśli są w kontekście)
        - Na końcu krótkie podsumowanie opcji

        TON:
        - jasny, medyczny, neutralny
        - bez emocjonalnych sformułowań

        KONTEKST:
        {context}

        PYTANIE:
        {query}

        ODPOWIEDŹ:
    """

    client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
    response = client.models.generate_content(
        model = "gemini-2.5-flash",
        contents = prompt,
    )

    return response.text

In [117]:
context = "\n\n".join(f"[DOC {i}] {doc}" for i, doc in enumerate(documents_rag))
model_response = rag_query(query, context)

In [118]:
display(Markdown(model_response))

Analiza objawów: Zgłasza Pan(i) silny ból głowy i potrzebę szybkiego uśmierzenia bólu, jednocześnie wskazując na uczulenie na salicylany.

Poniżej przedstawiono możliwe opcje leczenia bólu głowy dostępne w kontekście, z uwzględnieniem Pana(i) alergii:

*   **Alka-Seltzer®**
    *   **Działanie:** Substancja czynna, kwas acetylosalicylowy, działa przeciwbólowo, przeciwzapalnie i przeciwgorączkowo. Cytrynian sodu ma właściwości buforujące i zobojętnia nadmiar kwasu solnego w żołądku.
    *   **Wskazania:** Dolegliwości bólowe o lekkim i średnim nasileniu, np. bóle głowy, bóle mięśniowe, bóle zębów, ból i gorączka w przebiegu przeziębienia i grypy.
    *   **Przeciwwskazania:** Nie należy stosować leku Alka-Seltzer, jeśli pacjent ma uczulenie na substancję czynną - kwas acetylosalicylowy, inne salicylany lub którykolwiek z pozostałych składników tego leku.
    *   **Ocena dla Pana(i) przypadku:** Lek Alka-Seltzer zawiera kwas acetylosalicylowy, który jest salicylanem. Z uwagi na zgłoszone uczulenie na salicylany, ten lek jest przeciwwskazany.

*   **Almozen**
    *   **Działanie:** Almotryptan, substancja czynna leku Almozen, jest lekiem przeciwmigrenowym. Działa poprzez zwężenie naczyń krwionośnych w mózgu, co prowadzi do zmniejszenia reakcji zapalnej związanej z migreną.
    *   **Wskazania:** Stosowany jest w łagodzeniu bólów głowy związanych z napadami migreny z aurą lub bez aury.
    *   **Przeciwwskazania:** Nie przyjmować leku Almozen w przypadku uczulenia na almotryptan lub którykolwiek z pozostałych składników leku, choroby ograniczającej dopływ krwi do serca (np. zawał serca, ból w klatce piersiowej podczas aktywności/stresu, ciężkie/niekontrolowane nadciśnienie tętnicze), przebytego udaru mózgu lub zmniejszenia przepływu krwi do mózgu, niedrożności w dużych naczyniach krwionośnych ramion lub nóg, jednoczesnego przyjmowania innych leków przeciwmigrenowych (ergotamina, dihydroergotamina, metysergid, inni agoniści serotoniny), ciężkiej choroby wątroby.
    *   **Dawkowanie:** Zalecana dawka to jedna tabletka 12,5 mg, którą należy przyjąć najwcześniej jak to możliwe po wystąpieniu napadu migreny. Jeżeli napad migreny nie ustąpi, nie przyjmować więcej niż jednej tabletki podczas tego samego napadu. Jeśli wystąpi kolejny napad migreny w ciągu 24 godzin, można przyjąć drugą tabletkę w dawce 12,5 mg pod warunkiem zachowania przynajmniej 2-godzinnej przerwy. Maksymalna dawka dobowa to dwie tabletki (12,5 mg) na 24 godziny. Tabletkę należy połknąć popijając płynem, z posiłkiem lub niezależnie od posiłków.
    *   **Ocena dla Pana(i) przypadku:** Almozen jest lekiem przeznaczonym do leczenia bólów głowy związanych wyłącznie z napadami migreny. Nie zawiera salicylanów, co czyni go odpowiednim pod kątem Pana(i) alergii. Jeśli doświadczany ból głowy ma charakter migrenowy, ten lek może być rozważony.

*   **Allegra**
    *   **Działanie:** Allegra zawiera feksofenadyny chlorowodorek, który jest lekiem przeciwhistaminowym.
    *   **Wskazania:** Stosowana u dorosłych i młodzieży w wieku 12 lat i starszej w leczeniu objawów alergicznego zapalenia błony śluzowej nosa (np. kataru siennego), takich jak: kichanie, swędzenie nosa, katar lub uczucie zatkanego nosa oraz swędzenie, zaczerwienienie i łzawienie oczu.
    *   **Przeciwwskazania:** Jeśli pacjent ma uczulenie na feksofenadyny chlorowodorek lub którykolwiek z pozostałych składników leku.
    *   **Ocena dla Pana(i) przypadku:** Lek Allegra jest przeznaczony do leczenia objawów alergii i nie jest wskazany w leczeniu bólu głowy.

Podsumowanie opcji:
Z uwagi na uczulenie na salicylany, lek Alka-Seltzer jest przeciwwskazany. Lek Allegra nie jest przeznaczony do leczenia bólu głowy. Jedynym lekiem na ból głowy w dostarczonym kontekście, który nie zawiera salicylanów, jest Almozen. Jest on jednak wskazany wyłącznie w przypadku bólów głowy związanych z napadami migreny. W przypadku, gdy Pana(i) ból głowy nie jest migreną, w dostępnym kontekście brak jest innych leków wskazanych dla Pana(i) objawów z uwzględnieniem alergii na salicylany.